<center><img src='https://raw.githubusercontent.com/Jangrae/img/master/ml_python.png' width=600/></center>

<img src = "https://github.com/Jangrae/img/blob/master/diabetes.png?raw=true" width=800 align="left"/>

# 실습 내용

- Diabete 데이터로 모델링합니다.
- Logistic Regression 알고리즘으로 모델링합니다.


# 1.환경 준비

- 기본 라이브러리와 대상 데이터를 가져와 이후 과정을 준비합니다.

In [2]:
# 라이브러리 불러오기
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings(action='ignore')
%config InlineBackend.figure_format='retina'

In [3]:
# 데이터 읽어오기
path = 'https://raw.githubusercontent.com/jangrae/csv/master/diabetes.csv'
data = pd.read_csv(path)

# 2.데이터 이해

- 분석할 데이터를 충분히 이해할 수 있도록 다양한 탐색 과정을 수행합니다.

In [5]:
# 상위 몇 개 행 확인
data.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [6]:
# 기술통계 확인
data.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [7]:
# 범주값 개수 확인
data['Outcome'].value_counts()

Outcome
0    500
1    268
Name: count, dtype: int64

In [8]:
# 상관관계 확인
data.corr(numeric_only=True)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
Pregnancies,1.000000,0.129459,0.141282,-0.081672,-0.073535,0.017683,-0.033523,0.544341,0.221898
Glucose,0.129459,1.000000,0.152590,0.057328,0.331357,0.221071,0.137337,0.263514,0.466581
BloodPressure,0.141282,0.152590,1.000000,0.207371,0.088933,0.281805,0.041265,0.239528,0.065068
SkinThickness,-0.081672,0.057328,0.207371,1.000000,0.436783,0.392573,0.183928,-0.113970,0.074752
Insulin,-0.073535,0.331357,0.088933,0.436783,1.000000,0.197859,0.185071,-0.042163,0.130548
BMI,0.017683,0.221071,0.281805,0.392573,0.197859,1.000000,0.140647,0.036242,0.292695
DiabetesPedigreeFunction,-0.033523,0.137337,0.041265,0.183928,0.185071,0.140647,1.000000,0.033561,0.173844
Age,0.544341,0.263514,0.239528,-0.113970,-0.042163,0.036242,0.033561,1.000000,0.238356
Outcome,0.221898,0.466581,0.065068,0.074752,0.130548,0.292695,0.173844,0.238356,1.000000


# 3.데이터 준비

- 전처리 과정을 통해 머신러닝 알고리즘에 사용할 수 있는 형태의 데이터를 준비합니다.

**1) x, y 분리**

- 우선 target 변수를 명확히 지정합니다.
- target을 제외한 나머지 변수들 데이터는 x로 선언합니다.
- target 변수 데이터는 y로 선언합니다. 
- 이 결과로 만들어진 x는 데이터프레임, y는 시리즈가 됩니다.
- 이후 모든 작업은 x, y를 대상으로 진행합니다.

In [11]:
# target 확인
target = 'Outcome'

# 데이터 분리
x = data.drop(target, axis=1)
y = data.loc[:, target]

**2) 학습용, 평가용 데이터 분리**

- 학습용, 평가용 데이터를 적절한 비율로 분리합니다.
- 반복 실행 시 동일한 결과를 얻기 위해 random_state 옵션을 지정합니다.

In [18]:
# 모듈 불러오기
from sklearn.model_selection import train_test_split

# 7:3으로 분리
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=1)

# 4.모델링

- 본격적으로 모델을 선언하고 학습하고 평가하는 과정을 진행합니다.

In [33]:
# 1단계: 불러오기
from sklearn.linear_model import LogisticRegression  # 우리는 분류 문제를 풀고 있다. (Regression이라고 착각 ㄴㄴ/기존에 했던 것은 LinearRegression이었다.)
from sklearn.metrics import confusion_matrix, classification_report  # 분류문제 평가지표

임의로 선택된 초기 가중치를 선택    
그 가중치를 계속 업데이트    
    
최선의 가중치가 아니라고 생각하면 fitting(학습)할 때, warnning이 떨어질 수도 있다.   
필요하다면 반복할 수 있는 횟수를 지정해준다. (max_iter = 500)

In [35]:
# 2단계: 선언하기
model = LogisticRegression(max_iter = 500, random_state = 1)

In [37]:
# 3단계: 학습하기
model.fit(x_train, y_train)

LogisticRegression(max_iter=500, random_state=1)

In [40]:
# 4단계: 예측하기
y_pred = model.predict(x_test)

In [46]:
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

# 실제 1인데 1이라고 예측한 것 -> 0.58
# 0이라고 예측했는데 실제 0인것 ->0.79

[[132  14]
 [ 36  49]]
              precision    recall  f1-score   support

           0       0.79      0.90      0.84       146
           1       0.78      0.58      0.66        85

    accuracy                           0.78       231
   macro avg       0.78      0.74      0.75       231
weighted avg       0.78      0.78      0.78       231



# 5. 모델 살펴보기

* **시그모이드 함수**
$$ \huge p=\frac {1}{1+e^{-z}}  $$

In [55]:
# 회귀계수 확인
print(list(x))
print(model.coef_)
print(model.intercept_)

# 회귀 문제가 맞다는 것이 확인이 됨. 
# 변수 각각의 가중치와 편향이 확인이 되었다.

['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
[[ 0.10134046  0.03360035 -0.01571059 -0.00132708 -0.00069227  0.08955042
   0.54709395  0.01711044]]
[-7.86988557]


In [63]:
# 선형 판별식(f(x), z)
z = model.decision_function(x_test)   # 실제 회귀식의 값
print(z[10:21].round(2))

[-0.28 -2.51  3.93  0.98 -3.7   0.93 -1.18 -0.83 -1.9  -1.69 -0.49]


In [65]:
# 시그모이드 함수 사용
from scipy.special import expit
print(expit(z)[10:21].round(2))   # 위의 실제 회귀식의 값을 시그모이드를 통과하면 '확률값으로 바뀌게 된다.'

[0.43 0.07 0.98 0.73 0.02 0.72 0.24 0.3  0.13 0.16 0.38]


In [71]:
# 우리가 예상했던 값(위의 코드 확률과 비교했을 때 0.5이상인 것들이 1로 예측된 것을 확인할 수 있음.)
print(y_pred[10:21])

[0 0 1 1 0 1 0 0 0 0 0]


In [77]:
# 확률값 확인
p = model.predict_proba(x_test)
print(p[10:21].round(2))

# 왼쪽 값은 0일때의 확률, 오른쪽은 1일 때의 확률
# 우리는 1만 궁금하기 때문에 오른쪽만 봐도 된다.

[[0.57 0.43]
 [0.93 0.07]
 [0.02 0.98]
 [0.27 0.73]
 [0.98 0.02]
 [0.28 0.72]
 [0.76 0.24]
 [0.7  0.3 ]
 [0.87 0.13]
 [0.84 0.16]
 [0.62 0.38]]


# 6. 임계값 조정

In [88]:
# 확률값 얻기
p = model.predict_proba(x_test)

# 1에 대한 확률값만 뽑아보기(1의 확률 얻기)
p1 = p[:,1]   

# 확인
print(p[:10])
print(p1[:10])

[[0.58090402 0.41909598]
 [0.69548771 0.30451229]
 [0.85376328 0.14623672]
 [0.94485736 0.05514264]
 [0.79043253 0.20956747]
 [0.72141483 0.27858517]
 [0.64280931 0.35719069]
 [0.89572123 0.10427877]
 [0.83447558 0.16552442]
 [0.78795121 0.21204879]]
[0.41909598 0.30451229 0.14623672 0.05514264 0.20956747 0.27858517
 0.35719069 0.10427877 0.16552442 0.21204879]


In [92]:
# 임계값 = 0.5
y_pred2 = [1 if x > 0.5 else 0 for x in p1]
print(y_pred2[:10])

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [96]:
# 위에서 이미 했던 것을 수동으로 해본 것(위의 결과와 똑같음.)
print(classification_report(y_test, y_pred2))  

              precision    recall  f1-score   support

           0       0.79      0.90      0.84       146
           1       0.78      0.58      0.66        85

    accuracy                           0.78       231
   macro avg       0.78      0.74      0.75       231
weighted avg       0.78      0.78      0.78       231



In [100]:
# 임계값 = 0.45  -> 1을 더 많이 맞출 수 있도록(recall, accuracy가 높아진다.)
y_pred2 = [1 if x > 0.45 else 0 for x in p1]
print(classification_report(y_test, y_pred2))  

              precision    recall  f1-score   support

           0       0.80      0.90      0.85       146
           1       0.78      0.60      0.68        85

    accuracy                           0.79       231
   macro avg       0.79      0.75      0.76       231
weighted avg       0.79      0.79      0.79       231

